# Backtester Validation

This notebook runs a variety of toy examples to validate the correct functionality of the `backtest()` function.

Each example creates a purposely simple portfolio such that the correct outcomes can be computed by hand. The correct 
values are then compared with the outputs of the backtester.

Transaction costs follow the convention used throughout this project:
costs are tracked but considered as externally funded expenses. Therefore, transaction costs are recorded separately
to the portfolio's value. Consequently,

$$\text{Final Economic Value} = \text{Final Portfolio Value} - \text{Total Transaction Costs}.$$

In [54]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

#add the parent file to the module search path
sys.path.append(str(Path.cwd().parent))

#we can then find the quant_tools folder
from quant_tools.backtester import backtest

In [55]:
#this function generates a comparison table where we
#will compare the hand computations with the backtester output
def validation_table(expected, actual):
    expected = pd.Series(expected, dtype=float)
    actual = pd.Series(actual, dtype=float)

    comparison = pd.DataFrame({"Expected": expected, "Actual": actual,})

    comparison["Difference"] = comparison["Actual"] - comparison["Expected"]

    comparison["Matches"] = np.isclose(comparison["Actual"], comparison["Expected"])

    return comparison

## 1. All-Cash Portfolio

In this example, no investment instructions are supplied, so the portfolio should remain
entirely in cash. The stock returns must therefore have no effect on our net value.

$$V_{\text{final}} = V_0 = 10{,}000.$$

Because no assets are bought or sold, transaction costs should also be zero.

Expected:

- Final portfolio value: $10,000
- Transaction costs: $0
- Final economic value: $10,000

In [56]:
#we define a toy example of dates and stock returns, with target weights set to nans
dates = pd.date_range("2026-01-01", periods=3)

returns = pd.DataFrame({"AAPL": [0.10, -0.05, 0.20]}, index=dates)

target_weights = pd.DataFrame({"AAPL": [np.nan, np.nan, np.nan]}, index=dates)

#compute the output of the backtest function using these inputs
results = backtest(
    asset_returns=returns,
    target_weights=target_weights,
    initial_capital=10_000,
    cost_per_side=0.001
)

In [57]:
#display the results using the validation_table function
validation_table(
    expected={
        "Final Portfolio Value": 10_000,
        "Transaction Costs": 0,
        "Final Value": 10_000,
    },
    actual={
        "Final Portfolio Value":
            results["final_portfolio_value"],

        "Transaction Costs":
            results["total_transaction_costs"],

        "Final Value":
            results["final_value"],
    }
)

,Expected,Actual,Difference,Matches
Final Portfolio Value,10000.0,10000.0,0.0,True
Transaction Costs,0.0,0.0,0.0,True
Final Value,10000.0,10000.0,0.0,True


## 2. Single-Stock Buy-and-Hold

In this example, the portfolio invests 100% in AAPL on the first date and receives no
subsequent rebalance instructions.

The gross portfolio value should therefore be

$$10{,}000 \times 1.10 \times 0.95 = 10{,}450.$$

The initial purchase trades 100% of portfolio value. At a cost of 0.1%,

$$10{,}000 \times 0.001 = 10.$$

Because we assume transaction costs are externally funded, they do not reduce the
resulting value of the portfolio.

Expected:

- Final portfolio value: $10,450
- Transaction costs: $10
- Final economic value: $10,440

In [58]:
#again, toy inputs, then compute backtest on them
dates = pd.date_range("2026-01-01", periods=2)

returns = pd.DataFrame({"AAPL": [0.10, -0.05]}, index=dates)

target_weights = pd.DataFrame({"AAPL": [1.0, np.nan]}, index=dates)

results = backtest(
    asset_returns=returns,
    target_weights=target_weights,
    initial_capital=10_000,
    cost_per_side=0.001
)

In [59]:
validation_table(
    expected={
        "Final Portfolio Value": 10_450,
        "Transaction Costs": 10,
        "Final Value": 10_440,
    },
    actual={
        "Final Portfolio Value":
            results["final_portfolio_value"],

        "Transaction Costs":
            results["total_transaction_costs"],

        "Final Value":
            results["final_value"],
    }
)

,Expected,Actual,Difference,Matches
Final Portfolio Value,10450.0,10450.0,0.0,True
Transaction Costs,10.0,10.0,0.0,True
Final Value,10440.0,10440.0,0.0,True


## 3. Full Switch Between Assets

In this example, the portfolio first invests 100% in AAPL and then switches completely to
NVDA. Asset returns are set to zero so that only trading behaviour is tested.

On the first date, moving from cash to AAPL trades 100% of portfolio value:

$$\text{Traded Notional}_1 = 1.$$

On the second date, selling the entire AAPL position and buying an equal-sized
NVDA position gives

$$\text{Traded Notional}_2 = 1 + 1 = 2.$$

Using the conventional one-way turnover measure, turnover is 1 on both dates.

At a transaction cost of 0.1%, total costs should be

$$10{,}000(0.001)(1 + 2) = 30.$$

Expected final portfolio value is unchanged at $10,000, giving a final
economic value of $9,970.

In [60]:
dates = pd.date_range("2026-01-01", periods=2)

returns = pd.DataFrame({"AAPL": [0.0, 0.0], "NVDA": [0.0, 0.0]}, index=dates)

target_weights = pd.DataFrame({"AAPL": [1.0, 0.0], "NVDA": [0.0, 1.0]}, index=dates)

results = backtest(
    asset_returns=returns,
    target_weights=target_weights,
    initial_capital=10_000,
    cost_per_side=0.001
)

In [61]:
validation_table(
    expected={
        "Day 1 Traded Notional": 1.0,
        "Day 2 Traded Notional": 2.0,
        "Day 1 Turnover": 1.0,
        "Day 2 Turnover": 1.0,
        "Transaction Costs": 30.0,
        "Final Portfolio Value": 10_000,
        "Final Value": 9_970,
    },
    actual={
        "Day 1 Traded Notional":
            results["traded_notional"].iloc[0],

        "Day 2 Traded Notional":
            results["traded_notional"].iloc[1],

        "Day 1 Turnover":
            results["turnover"].iloc[0],

        "Day 2 Turnover":
            results["turnover"].iloc[1],

        "Transaction Costs":
            results["total_transaction_costs"],

        "Final Portfolio Value":
            results["final_portfolio_value"],

        "Final Value":
            results["final_value"],
    }
)

,Expected,Actual,Difference,Matches
Day 1 Traded Notional,1.0,1.0,0.0,True
Day 2 Traded Notional,2.0,2.0,0.0,True
Day 1 Turnover,1.0,1.0,0.0,True
Day 2 Turnover,1.0,1.0,0.0,True
Transaction Costs,30.0,30.0,0.0,True
Final Portfolio Value,10000.0,10000.0,0.0,True
Final Value,9970.0,9970.0,0.0,True


## 4. Weight Drift Without Rebalancing

The portfolio begins 50% in AAPL and 50% in NVDA.

During the first holding period:

- AAPL returns 0%
- NVDA returns 100%

Starting from $10,000, the two positions therefore become:

$$5{,}000 \quad\text{and}\quad 10{,}000.$$

Total portfolio value becomes $15,000, so the new weights before any
subsequent rebalance should be

$$w_{\text{AAPL}} = \frac{5{,}000}{15{,}000} = \frac13,$$

$$w_{\text{NVDA}} = \frac{10{,}000}{15{,}000} = \frac23.$$

The second target-weight row is entirely `NaN`, so the backtester should
retain these drifted weights rather than rebalance to 50/50.

In [62]:
dates = pd.date_range("2026-01-01", periods=2)

returns = pd.DataFrame({"AAPL": [0.0, 0.0], "NVDA": [1.0, 0.0]}, index=dates)

target_weights = pd.DataFrame({"AAPL": [0.5, np.nan], "NVDA": [0.5, np.nan]}, index=dates)

results = backtest(
    asset_returns=returns,
    target_weights=target_weights,
    initial_capital=10_000,
    cost_per_side=0.0
)

In [63]:
#we will compare portfolio weights on second day
second_day_weights = results["pre_rebalance_weights"].iloc[1]

validation_table(
    expected={
        "AAPL Weight": 1 / 3,
        "NVDA Weight": 2 / 3,
        "Final Portfolio Value": 15_000,
    },
    actual={
        "AAPL Weight":
            second_day_weights["AAPL"],

        "NVDA Weight":
            second_day_weights["NVDA"],

        "Final Portfolio Value":
            results["final_portfolio_value"],
    }
)

,Expected,Actual,Difference,Matches
AAPL Weight,0.333333,0.333333,0.0,True
NVDA Weight,0.666667,0.666667,0.0,True
Final Portfolio Value,15000.000000,15000.000000,0.0,True


## 5. Rebalancing After Weight Drift

In this example, the first holding period is identical to the previous test, so the portfolio
drifts from 50/50 to

$$\left(\frac13,\frac23\right).$$

On the second date, however, the target is again 50/50. The required trades are therefore

$$\Delta w_{\text{AAPL}} = \frac12-\frac13 = \frac16,$$

$$\Delta w_{\text{NVDA}} = \frac12-\frac23 = -\frac16.$$

Hence total traded notional is

$$\left|\frac16\right| + \left|-\frac16\right| = \frac13.$$

The conventional one-way turnover is half of this because no cash position
changes:

$$\text{Turnover} = \frac12 \times \frac13 = \frac16.$$

In [64]:
dates = pd.date_range("2026-01-01", periods=2)

returns = pd.DataFrame({"AAPL": [0.0, 0.0], "NVDA": [1.0, 0.0]}, index=dates)

target_weights = pd.DataFrame({"AAPL": [0.5, 0.5], "NVDA": [0.5, 0.5]}, index=dates)

results = backtest(
    asset_returns=returns,
    target_weights=target_weights,
    initial_capital=10_000,
    cost_per_side=0.0
)

In [65]:
#we will compare portfolio weights on second day
second_day_weights = results["pre_rebalance_weights"].iloc[1]

validation_table(
    expected={
        "Pre-Rebalance AAPL Weight": 1 / 3,
        "Pre-Rebalance NVDA Weight": 2 / 3,
        "Day 1 Traded Notional": 1.0,
        "Day 2 Traded Notional": 1 / 3,
        "Day 1 Turnover": 1.0,
        "Day 2 Turnover": 1 / 6,
    },
    actual={
        "Pre-Rebalance AAPL Weight":
            second_day_weights["AAPL"],

        "Pre-Rebalance NVDA Weight":
            second_day_weights["NVDA"],

        "Day 1 Traded Notional":
            results["traded_notional"].iloc[0],

        "Day 2 Traded Notional":
            results["traded_notional"].iloc[1],

        "Day 1 Turnover":
            results["turnover"].iloc[0],

        "Day 2 Turnover":
            results["turnover"].iloc[1],
    }
)

,Expected,Actual,Difference,Matches
Pre-Rebalance AAPL Weight,0.333333,0.333333,0.0,True
Pre-Rebalance NVDA Weight,0.666667,0.666667,0.0,True
Day 1 Traded Notional,1.000000,1.000000,0.0,True
Day 2 Traded Notional,0.333333,0.333333,0.0,True
Day 1 Turnover,1.000000,1.000000,0.0,True
Day 2 Turnover,0.166667,0.166667,0.0,True


## 6. Transaction Costs After Weight Drift

This example adds transaction costs to the previous rebalance example. The initial 50/50 purchase trades 100% of the $10,000 portfolio:

$$10{,}000 \times 1 \times 0.001 = 10.$$

After NVDA doubles, gross portfolio value becomes $15,000. Rebalancing the drifted weights back to 50/50 requires traded notional of

$$\frac13.$$

The second trading cost should therefore be

$$15{,}000 \times \frac13 \times 0.001 = 5.$$

Expected total transaction costs are therefore

$$10 + 5 = 15.$$

Because costs are considered separately, gross portfolio value remains $15,000. The final economic value is $14,985.

In [66]:
dates = pd.date_range("2026-01-01", periods=2)

returns = pd.DataFrame({"AAPL": [0.0, 0.0], "NVDA": [1.0, 0.0]}, index=dates)

target_weights = pd.DataFrame({"AAPL": [0.5, 0.5], "NVDA": [0.5, 0.5]}, index=dates)

results = backtest(
    asset_returns=returns,
    target_weights=target_weights,
    initial_capital=10_000,
    cost_per_side=0.001
)

In [67]:
validation_table(
    expected={
        "Day 1 Cost": 10.0,
        "Day 2 Cost": 5.0,
        "Total Costs": 15.0,
        "Final Portfolio Value": 15_000,
        "Final Value": 14_985,
    },
    actual={
        "Day 1 Cost":
            results["transaction_costs"].iloc[0],

        "Day 2 Cost":
            results["transaction_costs"].iloc[1],

        "Total Costs":
            results["total_transaction_costs"],

        "Final Portfolio Value":
            results["final_portfolio_value"],

        "Final Value":
            results["final_value"],
    }
)

,Expected,Actual,Difference,Matches
Day 1 Cost,10.0,10.0,0.0,True
Day 2 Cost,5.0,5.0,0.0,True
Total Costs,15.0,15.0,0.0,True
Final Portfolio Value,15000.0,15000.0,0.0,True
Final Value,14985.0,14985.0,0.0,True


## 7. Invalid Partial Target-Weight Row

For the backtest function, a target-weight row has an unambiguous interpretation only if:

- every asset weight is supplied, or
- the entire row is `NaN`, meaning no rebalance instruction.

In this example, a partially missing target row is ambiguous and should therefore raise a
`ValueError`.

In [68]:
dates = pd.date_range("2026-01-01", periods=1)

returns = pd.DataFrame({"AAPL": [0.0], "NVDA": [0.0]}, index=dates)

target_weights = pd.DataFrame({"AAPL": [0.5], "NVDA": [np.nan]}, index=dates)

error_raised = False
error_message = None

try:
    backtest(
        asset_returns=returns,
        target_weights=target_weights,
        initial_capital=10_000,
        cost_per_side=0.001
    )

except ValueError as error:
    error_raised = True
    error_message = str(error)

pd.Series({"ValueError Raised": error_raised, "Error Message": error_message})

ValueError Raised                                                 True
Error Message        Target-weight rows must be complete or entirel...
dtype: object

## 8. Return and Target-Weight Timing

Each return row represents the forward return earned after the target weights
for that same date are applied.

In this example, on the first date, the portfolio holds AAPL and AAPL returns 10%.

$$10{,}000 \times 1.10 = 11{,}000.$$

On the second date, the portfolio switches entirely to NVDA and NVDA returns 20%.

$$11{,}000 \times 1.20 = 13{,}200.$$

With transaction costs disabled for this test, the expected final portfolio
value is therefore $13,200.

The expected period returns recorded by the backtester are 10% and 20%.

In [69]:
dates = pd.date_range("2026-01-01", periods=2)

returns = pd.DataFrame({"AAPL": [0.10, 0.00], "NVDA": [0.00, 0.20]}, index=dates)

target_weights = pd.DataFrame({"AAPL": [1.0, 0.0], "NVDA": [0.0, 1.0]}, index=dates)

results = backtest(
    asset_returns=returns,
    target_weights=target_weights,
    initial_capital=10_000,
    cost_per_side=0.0
)

In [70]:
validation_table(
    expected={
        "First Period Return": 0.10,
        "Second Period Return": 0.20,
        "Final Portfolio Value": 13_200,
    },
    actual={
        "First Period Return":
            results["gross_returns"].iloc[0],

        "Second Period Return":
            results["gross_returns"].iloc[1],

        "Final Portfolio Value":
            results["final_portfolio_value"],
    }
)

,Expected,Actual,Difference,Matches
First Period Return,0.1,0.1,0.0,True
Second Period Return,0.2,0.2,0.0,True
Final Portfolio Value,13200.0,13200.0,0.0,True


## Validation Summary

The tests above validate the main accounting conventions used by the backtester:

- an all `NaN` target row means no rebalance instruction;
- positions earn only the returns of assets actually held;
- portfolio weights drift correctly as asset prices move;
- rebalancing trades are calculated from drifted weights rather than previous
  target weights;
- traded notional and conventional one-way turnover are distinguished;
- transaction costs scale with both portfolio value and traded notional;
- transaction costs follow the project's externally funded cost convention;
- partial target-weight rows are rejected as ambiguous;
- target weights and forward returns are aligned according to the intended
  period timing convention.

These hand-checkable cases provide a basic validation check for the backtesting engine. 
They are not exhaustive, but they cover the core accounting behaviour relied upon by the 
strategy research notebook.